In [14]:
from neo4j import GraphDatabase
import os
import json
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()  # will read .env in current dir
from neo4j import GraphDatabase
from openai import OpenAI
from pathlib import Path
from typing import List

In [15]:
uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
DB  = os.getenv("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 

driver = GraphDatabase.driver(uri, auth=(user, password))
print("[INFO] Connecting to:", uri)
print("[INFO] DB:", DB)
print("[INFO] USER:", user, "PWD:", password)
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"   # 1536 dims; use -large for 3072 if you prefer
oa_client = OpenAI(api_key=OPENAI_API_KEY)


[INFO] Connecting to: neo4j+s://62b9e173.databases.neo4j.io
[INFO] DB: neo4j
[INFO] USER: neo4j PWD: oHiXwOTSqk3Tv9TkX-wrcngsjffZeIWIYkcvwwzBefQ


In [18]:
# ingest_docs_with_embeddings_skip_empty.py
# pip install neo4j openai numpy

import os, json, re, unicodedata
from typing import List, Dict, Any, Tuple
from pathlib import Path

import numpy as np
from neo4j import GraphDatabase, basic_auth
from openai import OpenAI

# ========= CONFIG =========
URI = os.environ.get("NEO4J_URI", "neo4j://localhost:7687")
USER = os.environ.get("NEO4J_USERNAME", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD", "password")
DB = os.environ.get("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
EMBED_MODEL = os.environ.get("EMBED_MODEL", "text-embedding-3-small")
BATCH_SIZE = int(os.environ.get("EMBED_BATCH_SIZE", "128"))
MAX_CHARS_PER_INPUT = int(os.environ.get("MAX_CHARS_PER_INPUT", "12000"))
MIN_NONWS_CHARS = int(os.environ.get("MIN_NONWS_CHARS", "1"))  # skip if < this after cleaning

driver = GraphDatabase.driver(URI, auth=basic_auth(USER, PASSWORD))
oa_client = OpenAI(api_key=OPENAI_API_KEY)

def verify():
    print("[INFO] Connecting to:", URI)
    print("[INFO] DB:", DB)
    print("[INFO] USER:", USER)
    driver.verify_connectivity()
    print("[INFO] Connected. Server info:", driver.get_server_info())

# ========= SANITIZE =========
CONTROL_CHARS_RE = re.compile(r"[\u0000-\u0008\u000B\u000C\u000E-\u001F\u007F]")

def sanitize_for_embedding(x) -> str:
    if x is None:
        s = ""
    else:
        s = str(x)
    s = unicodedata.normalize("NFKC", s)
    s = CONTROL_CHARS_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > MAX_CHARS_PER_INPUT:
        s = s[:MAX_CHARS_PER_INPUT].rstrip()
    return s

# ========= IO =========
def read_docs_from_json(file_path: str) -> List[Dict[str, Any]]:
    p = Path(file_path)
    content = p.read_text(encoding="utf-8").strip()
    if not content:
        return []
    if content.lstrip().startswith("["):
        docs = json.loads(content)
    else:
        docs = [json.loads(line) for line in content.splitlines() if line.strip()]
    out = []
    for d in docs:
        _id = d.get("_id")
        title = d.get("title")
        text = d.get("text")
        if text is None:
            text = ""
        if not isinstance(text, str):
            text = json.dumps(text, ensure_ascii=False)
        if not isinstance(title, str):
            title = "" if title is None else str(title)
        out.append({"_id": _id, "title": title, "text": text})
    return out

# ========= EMBEDDINGS (skip empty) =========
def embed_texts_indexed(texts: List[str], model: str = EMBED_MODEL, batch_size: int = BATCH_SIZE) -> Dict[int, List[float]]:
    """
    Returns a dict: original_index -> embedding (list[float])
    Skips items that are empty/too short after sanitization.
    """
    # Build (idx, clean_text) for non-empty
    pairs: List[Tuple[int, str]] = []
    skipped = []
    for idx, t in enumerate(texts):
        clean = sanitize_for_embedding(t)
        if len(clean) >= MIN_NONWS_CHARS:
            pairs.append((idx, clean))
        else:
            skipped.append(idx)

    print(f"[INFO] Valid texts: {len(pairs)} | Skipped (empty/too short): {len(skipped)}")

    embeds_by_idx: Dict[int, List[float]] = {}
    for start in range(0, len(pairs), batch_size):
        chunk = pairs[start:start+batch_size]
        chunk_texts = [c[1] for c in chunk]
        try:
            resp = oa_client.embeddings.create(model=model, input=chunk_texts)
            for (orig_idx, _), item in zip(chunk, resp.data):
                embeds_by_idx[orig_idx] = item.embedding
        except Exception:
            # Per-item fallback to pinpoint bad one (rare after sanitation)
            for orig_idx, one_text in chunk:
                try:
                    resp1 = oa_client.embeddings.create(model=model, input=[one_text])
                    embeds_by_idx[orig_idx] = resp1.data[0].embedding
                except Exception as e1:
                    raise ValueError(
                        f"[EMBEDDING ERROR] Bad sanitized input at original index {orig_idx}. repr={repr(one_text)[:200]}"
                    ) from e1
    return embeds_by_idx

# ========= Neo4j =========
def ensure_constraints_and_index(dim: int):
    with driver.session(database=DB) as session:
        session.run("""
        CREATE CONSTRAINT doc_id IF NOT EXISTS
        FOR (d:Document)
        REQUIRE d.id IS UNIQUE
        """)
        try:
            session.run(f"""
            CREATE VECTOR INDEX doc_text_embedding IF NOT EXISTS
            FOR (d:Document) ON (d.embedding)
            OPTIONS {{
              indexConfig: {{
                `vector.dimensions`: {dim},
                `vector.similarity_function`: 'cosine'
              }}
            }}
            """)
        except Exception as e:
            print("[WARN] Skipping vector index (not supported on this DB/tier):", e)

def upsert_documents_with_embeddings(rows: List[dict]):
    cypher = """
    UNWIND $rows AS r
    MERGE (n:Document {id: r.id})
    SET n.title = r.title,
        n.text  = r.text,
        n.embedding = r.embedding
    """
    with driver.session(database=DB) as session:
        session.run(cypher, rows=rows)

# ========= MAIN =========
if __name__ == "__main__":
    verify()

    file_path = os.environ.get(
        "DOCS_PATH",
        "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/datasets/nq/hybrid.jsonl"
    )
    docs = read_docs_from_json(file_path)
    print(f"[INFO] Loaded {len(docs)} docs")

    if not docs:
        print("[WARN] No documents found. Exiting.")
        raise SystemExit(0)

    texts = [d["text"] for d in docs]
    print("[INFO] Embedding texts (skipping empty)…")
    embeds_by_idx = embed_texts_indexed(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE)

    if not embeds_by_idx:
        print("[WARN] No embeddings produced (all texts empty?). Exiting.")
        raise SystemExit(0)

    # Prepare rows only for docs that got an embedding
    rows = []
    for i, d in enumerate(docs):
        emb = embeds_by_idx.get(i)
        if emb and d.get("_id"):
            rows.append({
                "id": d["_id"],
                "title": d["title"],
                "text": sanitize_for_embedding(d["text"]),  # store cleaned version if you like
                "embedding": emb,
            })

    if not rows:
        print("[WARN] Nothing to upsert (no valid _id or embeddings). Exiting.")
        raise SystemExit(0)

    dim = len(rows[0]["embedding"])
    print(f"[INFO] Vector dimension: {dim}")
    print(f"[INFO] Upserting {len(rows)} Document nodes with embeddings …")

    ensure_constraints_and_index(dim)
    upsert_documents_with_embeddings(rows)

    print("[INFO] Done.")


[INFO] Connecting to: neo4j+s://62b9e173.databases.neo4j.io
[INFO] DB: neo4j
[INFO] USER: neo4j
[INFO] Connected. Server info: <neo4j.api.ServerInfo object at 0x7d67c79ccb80>
[INFO] Loaded 1001 docs
[INFO] Embedding texts (skipping empty)…
[INFO] Valid texts: 1000 | Skipped (empty/too short): 1
[INFO] Vector dimension: 1536
[INFO] Upserting 1000 Document nodes with embeddings …
[INFO] Done.
